# GLV Candidate Benchmark Analysis

Compares the experimental candidate branches for one GLV against their **baseline**, deciding
wins on the **medium wall-clock latency median** — the primary signal for this latency-first run.

This run is **medium-only** and **latency-first**. The medium cell is a deserialization-dominated
query (~200766 objects, ~8 s/req cross-region, so network RTT is <1% of the wall clock). Each arm
runs the medium cell across 3 sweeps and the harness appends one ledger row per sweep. The
representative latency for an arm is the **median of its per-sweep `median` values** (median-of-medians).

**Inputs** are pulled from S3 (`s3://kirill-tp-benchmarks/cand-results/<glv>/`), where the EC2
client published `~/cand-results` after the sweep. Each arm has its own `ledger.csv`.

**The decision:** a candidate is **interesting** if its medium latency beats the baseline
(improvement > `LATENCY_IMPROVE_MIN`); a candidate is a **WIN** if that improvement also reproduces
across the per-sweep medians AND `status=ok`/`errors=0` on every row.

**yappi CPU self-time** is retained as an *optional* deterministic attribution cross-check: this run
did **not** publish `medium-s*-yappi-cpu.txt` profiles (yappi was dropped), so those sections are
skipped cleanly. There is **no tiny cell** in this run. Restart kernel → Run All is fully
reproducible: every input comes from S3 + the parameters cell.

## 0. Parameters

Edit these to retarget another GLV or run.

In [ ]:
GLV               = 'python'
BUCKET            = 'kirill-tp-benchmarks'
PREFIX            = f'cand-results/{GLV}'        # S3 prefix the EC2 client synced to
BASELINE          = 'bench-baseline'             # baseline arm = its branch-tag directory

# --- PRIMARY signal: medium wall-clock latency -------------------------------
# A candidate is "interesting" when its median latency beats baseline. 0.0 == any
# improvement counts; a *robust* win should also reproduce across the per-sweep
# medians (see the verdict). Raise this to demand a minimum effect size.
LATENCY_IMPROVE_MIN = 0.0                         # min (base-cand)/base improvement to flag

# --- OPTIONAL cross-check: yappi CPU self-time -------------------------------
# Only used if medium-s*-yappi-cpu.txt profiles exist for the arms. This run did
# not publish them, so the yappi sections are skipped cleanly.
CPU_GATE          = 0.05                          # optional: total self-time drop >= 5% in every sweep

SWEEPS            = [1, 2, 3]
LOCAL_DIR         = 'cand-results-data'           # where S3 objects are mirrored locally

## 1. Setup

In [ ]:
!pip install pandas plotly boto3 nbformat -q

import os, re, json
import boto3
import pandas as pd
import plotly.graph_objects as go

PASS_COLOR, FAIL_COLOR, BASE_COLOR = '#00CC96', '#BBBBBB', '#636EFA'
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

## 2. Download from S3

Mirrors every `ledger.csv` and (when present) `medium-s*-yappi-cpu.txt` under the prefix
into a local tree `LOCAL_DIR/<arm>/...`. The yappi profiles are optional; this run
has none. Idempotent — safe to re-run.

In [ ]:
def download_prefix(bucket, prefix, local_dir):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    n = 0
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            key = obj['Key']
            base = os.path.basename(key)
            # ledger.csv is required; yappi profiles are optional (kept so this still
            # works if profiles are ever published again) and never required below.
            if base != 'ledger.csv' and not re.match(r'medium-s\d+-yappi-cpu\.txt$', base):
                continue
            rel = key[len(prefix):].lstrip('/')
            dest = os.path.join(local_dir, rel)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            s3.download_file(bucket, key, dest)
            n += 1
    return n

count = download_prefix(BUCKET, PREFIX, LOCAL_DIR)
print(f'Downloaded {count} file(s) from s3://{BUCKET}/{PREFIX} -> {LOCAL_DIR}/')
arms = sorted(d for d in os.listdir(LOCAL_DIR)
              if os.path.isdir(os.path.join(LOCAL_DIR, d)) and not d.startswith('_'))
print('Arms found:', arms)

# Did any yappi profiles come down? Drives the OPTIONAL cross-check below.
HAS_YAPPI = any(
    re.match(r'medium-s\d+-yappi-cpu\.txt$', f)
    for arm in arms
    for f in os.listdir(os.path.join(LOCAL_DIR, arm))
)
print('yappi profiles present:', HAS_YAPPI,
      '(latency-only analysis)' if not HAS_YAPPI else '(latency + optional CPU cross-check)')

assert BASELINE in arms, (
    f'baseline {BASELINE!r} not among arms found in {LOCAL_DIR}/: {arms}. '
    f'Set BASELINE in the parameters cell to one of these directory tags.')

## 3. Load ledgers

One tidy DataFrame, one row per cell, with an `arm` column. We keep only `candidate-eval` rows.

In [ ]:
def load_ledgers(local_dir, arms):
    frames = []
    for arm in arms:
        p = os.path.join(local_dir, arm, 'ledger.csv')
        if not os.path.exists(p):
            print(f'WARN: no ledger for {arm}'); continue
        d = pd.read_csv(p)
        d['arm'] = arm
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    if 'label' in df.columns:
        df = df[df['label'] == 'candidate-eval'].copy()
    df['is_baseline'] = df['arm'] == BASELINE
    return df

ledger = load_ledgers(LOCAL_DIR, arms)
print(f'{len(ledger)} candidate-eval rows across {ledger["arm"].nunique()} arms')
print('point_value values:', sorted(ledger['point_value'].dropna().unique().tolist()))
ledger[['arm','point_value','status','errors','median','git_sha','host']].sort_values(['arm','point_value'])

## 4. PRIMARY signal — medium wall-clock latency (the decision)

This is the latency-first decision. From the ledger we take the `point_value=='medium'` rows
(the harness appended one per sweep, so each arm has ~`len(SWEEPS)` rows). Each row's `median` is
sec/req for that sweep (clean wall-clock — these cells were **unprofiled**).

For each arm we compute the **median of its per-sweep medians** (median-of-medians) as the
representative latency, plus the per-sweep medians for a reproducibility check. A candidate's
improvement vs baseline is `(base_med - cand_med) / base_med`. A candidate is flagged **interesting**
if `improvement > LATENCY_IMPROVE_MIN`, and is **reproducible** if it also beats baseline's per-sweep
median in *every* matched sweep.

In [ ]:
# Medium rows only (label already filtered to candidate-eval in load_ledgers).
med = ledger[ledger['point_value'] == 'medium'].copy()
med['median'] = pd.to_numeric(med['median'], errors='coerce')
assert not med.empty, "no point_value=='medium' rows found in the ledger"

# Representative latency per arm = median of its per-sweep medians (median-of-medians).
lat_med = med.groupby('arm')['median'].median()
# Per-arm sorted list of per-sweep medians, for the reproducibility check.
per_sweep = med.groupby('arm')['median'].apply(lambda s: sorted(s.dropna().tolist()))

base_med = lat_med.get(BASELINE)
base_sweeps = per_sweep.get(BASELINE, [])
assert base_med is not None and pd.notna(base_med), f'no medium latency for baseline {BASELINE!r}'

def reproducible(cand_sweeps):
    """True if the candidate beats baseline's per-sweep medians in every matched sweep
    (compared rank-by-rank on the sorted per-sweep medians)."""
    if not cand_sweeps or not base_sweeps:
        return False
    k = min(len(cand_sweeps), len(base_sweeps))
    return all(cand_sweeps[i] < base_sweeps[i] for i in range(k))

cand_arms = [a for a in arms if a != BASELINE]
lat_rows = []
for a in cand_arms:
    cm = lat_med.get(a)
    improve = (base_med - cm) / base_med if pd.notna(cm) else None
    lat_rows.append({
        'candidate': a,
        'cand_med (s/req)': cm,
        'base_med (s/req)': base_med,
        'improvement': improve,
        'interesting': (improve is not None) and (improve > LATENCY_IMPROVE_MIN),
        'reproducible': reproducible(per_sweep.get(a, [])),
        'n_sweeps': len(per_sweep.get(a, [])),
    })
latency = pd.DataFrame(lat_rows).sort_values('improvement', ascending=False).reset_index(drop=True)

print(f'{GLV}: medium latency vs baseline {BASELINE!r}  (base median-of-medians = {base_med:.4f} s/req)')
print('INTERESTING:', latency.loc[latency['interesting'], 'candidate'].tolist() or 'none')

_show = latency.copy()
_show['improvement'] = _show['improvement'].map(lambda v: f'{v*100:+.2f}%' if pd.notna(v) else '—')
_show

In [ ]:
# Medium latency per arm (median-of-medians), baseline highlighted. Lower = better.
order = [BASELINE] + latency['candidate'].tolist()
_interesting = latency.set_index('candidate')['interesting']
fig = go.Figure()
fig.add_bar(
    x=[a.replace('auto/','') for a in order],
    y=[lat_med.get(a) for a in order],
    marker_color=[BASE_COLOR if a == BASELINE else
                  (PASS_COLOR if bool(_interesting.get(a, False)) else FAIL_COLOR)
                  for a in order],
    text=[f'{lat_med.get(a):.3f}' if pd.notna(lat_med.get(a)) else '' for a in order],
    textposition='outside',
)
fig.add_hline(y=base_med, line_dash='dash', line_color=BASE_COLOR,
              annotation_text=f'baseline {base_med:.3f} s/req', annotation_position='top left')
fig.update_layout(
    title=f'{GLV}: medium latency per arm (median of {len(SWEEPS)} sweeps, lower = faster; '
          f'baseline grey, green = beats baseline)',
    yaxis_title='median sec/req', height=460, showlegend=False)
fig.show()

## 5. OPTIONAL cross-check — yappi CPU self-time (`tsub`)

> **Skipped when no profiles exist.** This is a deterministic attribution cross-check, **not** the
> primary decision. This run did not publish `medium-s*-yappi-cpu.txt`, so every cell in this section
> is guarded by `HAS_YAPPI` and degrades to a clear "no yappi profiles found" message. The careful
> parser below is preserved verbatim for runs that *do* publish profiles.

When profiles exist, the hook writes a **func-stats** table (cols `name ncall tsub ttot tavg`, sorted
by `tsub desc`) after two header lines, then appends a separate **THREAD STATS** table
(cols `name id tid ttot scnt`). The cross-check metric is the **sum of the func-stats `tsub` column**.

Two gotchas the parser handles: (1) *both* tables' header rows start with `name`, so we only start
on the func-stats header (the one containing `tsub`); (2) we **stop at the `==== THREAD STATS ====`
separator** — otherwise the thread table's giant `tid` values leak into the sum and produce a
nonsensical ~1e14 total. `tsub` is read as the 3rd-from-last field (the `name` column may contain
spaces, but the trailing four numeric columns are fixed).

In [ ]:
def sum_tsub(path):
    """Sum the tsub column of a yappi -yappi-cpu.txt file. Returns float or None.

    The hook writes a FUNC-stats table (cols: name ncall tsub ttot tavg) and then
    appends a THREAD STATS table (cols: name id tid ttot scnt). Both header rows
    start with 'name', so we (a) only start on the func-stats header — the one that
    also contains 'tsub' — and (b) STOP at the '==== THREAD STATS ====' separator.
    Summing into the thread table would add giant thread ids (tid) and produce
    nonsense totals (~1e14).
    """
    if not os.path.exists(path):
        return None
    total, started = 0.0, False
    with open(path) as fh:
        for line in fh:
            s = line.rstrip('\n')
            if s.startswith('==== THREAD STATS'):   # end of the func-stats table
                break
            if not started:
                if s.lstrip().startswith('name') and 'tsub' in s:  # func-stats header
                    started = True
                continue
            if not s.strip():
                continue
            parts = s.split()
            if len(parts) < 4:
                continue
            try:                                     # ... ncall tsub ttot tavg -> tsub = parts[-3]
                total += float(parts[-3])
            except ValueError:
                continue
    return round(total, 3)

# OPTIONAL: only build the tsub frame when profiles were actually downloaded.
if not HAS_YAPPI:
    tsub = pd.DataFrame(columns=['arm', 'sweep', 'total_tsub'])
    print('no yappi profiles found — latency-only analysis (skipping CPU self-time cross-check)')
else:
    tsub = pd.DataFrame([
        {'arm': arm, 'sweep': s,
         'total_tsub': sum_tsub(os.path.join(LOCAL_DIR, arm, f'medium-s{s}-yappi-cpu.txt'))}
        for arm in arms for s in SWEEPS
    ])
    missing = tsub[tsub['total_tsub'].isna()]
    if len(missing):
        print('WARN: missing/empty profiles:'); print(missing.to_string(index=False))
    # sanity: a medium yappi run should total on the order of tens-to-hundreds of seconds,
    # never ~1e14 — that would mean the thread table leaked in (see the docstring).
    hi = tsub['total_tsub'].dropna()
    if len(hi) and hi.max() > 1e6:
        print(f'WARN: implausible total_tsub (max={hi.max():.3g}) — check the parser/profile format')

tsub.pivot(index='arm', columns='sweep', values='total_tsub') if not tsub.empty else tsub

### 5a. Optional CPU self-time drop vs baseline

> Computed **only** when profiles exist (`HAS_YAPPI`). For each candidate arm and sweep:
> `drop = (base - cand) / base`. The cross-check flags an arm when its **minimum** drop across all
> sweeps is ≥ `CPU_GATE` — a CPU win that reproduces in every sweep. This is supporting attribution
> for the latency decision, not the decision itself.

In [ ]:
def show_pct(df, cols):
    out = df.copy()
    for c in cols:
        out[c] = out[c].map(lambda v: f'{v*100:+.1f}%' if pd.notna(v) else '—')
    return out

if not HAS_YAPPI:
    gate = pd.DataFrame(columns=['arm'] + [f's{s}_drop' for s in SWEEPS] +
                                ['min_drop', 'mean_drop', 'PASS'])
    print('no yappi profiles found — skipping the optional CPU self-time cross-check')
else:
    base_by_sweep = tsub[tsub['arm'] == BASELINE].set_index('sweep')['total_tsub']

    def drop_for(arm, s):
        cand = tsub[(tsub['arm']==arm) & (tsub['sweep']==s)]['total_tsub']
        b = base_by_sweep.get(s)
        if cand.empty or pd.isna(cand.iloc[0]) or b is None or pd.isna(b):
            return None
        return (b - cand.iloc[0]) / b

    gate_rows = []
    for arm in [a for a in arms if a != BASELINE]:
        drops = {s: drop_for(arm, s) for s in SWEEPS}
        vals = [d for d in drops.values() if d is not None]
        rec = {'arm': arm, **{f's{s}_drop': drops[s] for s in SWEEPS}}
        rec['min_drop'] = min(vals) if vals else None
        rec['mean_drop'] = sum(vals)/len(vals) if vals else None
        rec['PASS'] = (rec['min_drop'] is not None) and (rec['min_drop'] >= CPU_GATE)
        gate_rows.append(rec)
    gate = pd.DataFrame(gate_rows)

    print(f'Optional CPU cross-check: total self-time drop >= {CPU_GATE*100:.0f}% in EVERY sweep')

show_pct(gate, [f's{s}_drop' for s in SWEEPS] + ['min_drop','mean_drop']) if not gate.empty else gate

## 6. Validity gates — any failure invalidates the arm

- **status** must be `ok` on every row.
- **errors** must be all `0` on every row.

There is no tiny cell in this latency-first run, so the old tiny-regression guard does not apply.

> **Result-count guard:** the per-execution graph result count (must match the baseline, e.g. 200766
> objects for the medium `times(12)` traversal) lives in the per-cell logs under `<arm>/logs/`, which
> this notebook does **not** sync from S3 (only `ledger.csv` + optional profiles). The ledger's
> `count` column is the number of *measurements*, not graph results. Verify the medium result count on
> the EC2 box (runbook §5c) or extend the S3 sync to include `logs/` if you want it here.

In [ ]:
def errs_ok(v):
    try:
        return all(int(x) == 0 for x in json.loads(v))
    except Exception:
        return v in (0, '0', None)

guard = ledger.groupby('arm').agg(
    all_status_ok=('status', lambda s: (s == 'ok').all()),
).reset_index()
guard['errors_ok'] = guard['arm'].map(
    ledger.groupby('arm')['errors'].apply(lambda col: all(errs_ok(v) for v in col)))
guard['valid'] = guard['all_status_ok'] & guard['errors_ok']
guard

## 7. Optional yappi charts

The primary latency chart is in §4. The charts below are the **optional** CPU self-time cross-check
and render only when profiles exist; otherwise they print a skip message.

In [ ]:
# 7a. CPU self-time drop per candidate (mean across sweeps; error bars span min..max), gate line.
if not HAS_YAPPI or gate.empty:
    print('no yappi profiles found — skipping CPU self-time drop chart')
else:
    cand_arms_y = [a for a in arms if a != BASELINE]
    def grow(a): return gate[gate['arm']==a].iloc[0]
    means = [grow(a)['mean_drop'] for a in cand_arms_y]
    mins  = [grow(a)['min_drop']  for a in cand_arms_y]
    sweep_vals = lambda a: [grow(a)[f's{s}_drop'] for s in SWEEPS if pd.notna(grow(a)[f's{s}_drop'])]
    maxs  = [max(sweep_vals(a)) if sweep_vals(a) else None for a in cand_arms_y]
    colors = [PASS_COLOR if bool(grow(a)['PASS']) else FAIL_COLOR for a in cand_arms_y]

    def err_plus(m, mx):  return (mx - m)*100 if (pd.notna(m) and mx is not None) else 0
    def err_minus(m, mn): return (m - mn)*100 if (pd.notna(m) and mn is not None) else 0

    fig = go.Figure()
    fig.add_bar(
        x=[a.replace('auto/','') for a in cand_arms_y],
        y=[(m*100 if pd.notna(m) else 0) for m in means],
        marker_color=colors,
        error_y=dict(type='data', symmetric=False,
                     array=[err_plus(m, mx) for m, mx in zip(means, maxs)],
                     arrayminus=[err_minus(m, mn) for m, mn in zip(means, mins)]),
        text=[f'{m*100:+.1f}%' if pd.notna(m) else '—' for m in means], textposition='outside',
    )
    fig.add_hline(y=CPU_GATE*100, line_dash='dash', line_color='red',
                  annotation_text=f'{CPU_GATE*100:.0f}% cross-check', annotation_position='top left')
    fig.update_layout(title=f'{GLV}: CPU self-time drop vs baseline (optional cross-check, mean of {len(SWEEPS)} sweeps)',
                      yaxis_title='self-time drop % (higher = faster)', height=460, showlegend=False)
    fig.show()

In [ ]:
# 7b. Total tsub per arm per sweep — shows reproducibility across sweeps.
if not HAS_YAPPI or tsub.empty:
    print('no yappi profiles found — skipping total self-time per-sweep chart')
else:
    fig = go.Figure()
    for s in SWEEPS:
        sub = tsub[tsub['sweep'] == s].set_index('arm')['total_tsub']
        fig.add_bar(name=f'sweep {s}',
                    x=[a.replace('auto/','') for a in arms],
                    y=[sub.get(a) for a in arms])
    fig.update_layout(barmode='group',
                      title=f'{GLV}: total yappi self-time per arm per sweep (lower = faster)',
                      yaxis_title='sum tsub (s)', height=460)
    fig.show()

## 8. Verdict

Auto-filled decision table. The primary signal is **medium wall-clock latency**: a candidate is a
**WIN** when it beats the baseline latency (`improvement > LATENCY_IMPROVE_MIN`), that improvement
**reproduces** across the per-sweep medians, and it passes the **status/errors** validity gates.

The **optional CPU drop** column appears only when yappi profiles were published (`HAS_YAPPI`); it is
supporting attribution and does not change the verdict. (Confirm the medium result-count guard
separately — see §6.)

In [ ]:
lat_by_cand = latency.set_index('candidate')
guard_by_arm = guard.set_index('arm')
gate_by_arm = gate.set_index('arm') if not gate.empty else None

rows = []
for a in cand_arms:
    lr = lat_by_cand.loc[a] if a in lat_by_cand.index else None
    improve = lr['improvement'] if lr is not None else None
    interesting = bool(lr['interesting']) if lr is not None else False
    repro = bool(lr['reproducible']) if lr is not None else False

    gd = guard_by_arm.loc[a] if a in guard_by_arm.index else None
    status_ok = bool(gd['valid']) if gd is not None else False

    win = interesting and repro and status_ok

    row = {
        'candidate': a.replace('auto/',''),
        'latency improve': f'{improve*100:+.2f}%' if (improve is not None and pd.notna(improve)) else '—',
        'reproducible': repro,
        'status/errors ok': status_ok,
        'VERDICT': 'WIN' if win else 'no',
    }
    # OPTIONAL extra column, only when profiles exist.
    if gate_by_arm is not None and a in gate_by_arm.index:
        md = gate_by_arm.loc[a]['min_drop']
        row['CPU drop (opt)'] = f'{md*100:+.1f}%' if pd.notna(md) else '—'
    rows.append(row)

verdict = pd.DataFrame(rows)
print(f'GLV={GLV}  baseline={BASELINE}  primary=medium wall-clock latency  '
      f'(improve > {LATENCY_IMPROVE_MIN*100:.1f}% + reproducible + status/errors ok)')
print('yappi CPU cross-check:', 'shown (profiles present)' if HAS_YAPPI else 'absent (latency-only)')
print('WINNERS:', [r['candidate'] for r in rows if r['VERDICT']=='WIN'] or 'none')
verdict